# Train YOLO11 signature detector on SignverOD (GPU)

Before running: **Runtime -> Change runtime type -> T4 GPU** (or whatever GPU is offered), then Save.

**Important**: `/content` is the Colab VM's local disk — if the runtime disconnects (idle timeout, session limit, network blip), everything under `/content` is wiped with no recovery. A 3-hour training run finishing and then sitting for a bit before you come back to download it is exactly the kind of gap that triggers a disconnect. This version writes checkpoints directly to **Google Drive** as training runs (`last.pt`/`best.pt` are updated every epoch regardless of `save_period`), so a mid-training disconnect loses at most the current epoch, not the whole run — and there's no separate "remember to download it" step to forget.

Steps: mount Drive -> install ultralytics -> upload `signverod_yolo.zip` -> train `yolo11n` (writing straight to Drive) -> optionally train `yolo11s` / compare -> weights are already safe on Drive; browser download is optional.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE_RUNS_ROOT = pathlib.Path('/content/drive/MyDrive/signature_training/runs')
DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print('Checkpoints will be written under:', DRIVE_RUNS_ROOT)

In [ ]:
!pip install -q ultralytics
import torch
print('CUDA available:', torch.cuda.is_available(), '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU (check Runtime > Change runtime type)')

## Upload the dataset
Upload `signverod_yolo.zip` (~390MB, from `training/data/signverod_yolo.zip` in your local project).

In [ ]:
from google.colab import files
uploaded = files.upload()  # select signverod_yolo.zip
zip_name = next(iter(uploaded))
print('uploaded:', zip_name)

In [ ]:
import zipfile, pathlib

DATA_ROOT = pathlib.Path('/content/signverod_yolo')
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content')

yaml_path = DATA_ROOT / 'signature.yaml'
yaml_path.write_text(
    f"path: {DATA_ROOT}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "names:\n"
    "  0: signature\n"
)
print(yaml_path.read_text())
print('train images:', len(list((DATA_ROOT/'images/train').glob('*'))))
print('val images:', len(list((DATA_ROOT/'images/val').glob('*'))))
print('test images:', len(list((DATA_ROOT/'images/test').glob('*'))))

## Train yolo11n
Same config as `training/train_signature.py --model yolo11n --run-name n_baseline`, on GPU (`device=0`), writing
straight to Drive via `project=DRIVE_RUNS_ROOT/'yolo11n'` — survives a runtime disconnect mid-run.

In [ ]:
from ultralytics import YOLO

model_n = YOLO('yolo11n.pt')
results_n = model_n.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    project=str(DRIVE_RUNS_ROOT / 'yolo11n'),
    name='n_baseline',
    device=0,
)
n_best = results_n.save_dir / 'weights' / 'best.pt'
print('yolo11n best weights (on Drive):', n_best)
print('exists:', n_best.exists())

## Download yolo11n weights to your browser
Run this right after the training cell above finishes, in the same session — the file is already safe on Drive
either way, but this saves you a trip through Drive's UI. If the runtime *has* disconnected by the time you get
here, re-run the Drive-mount cell (first cell) and this cell will still find the file on Drive.

In [ ]:
from google.colab import files
files.download(str(n_best))

## (Optional, later) Train yolo11s
Separate, explicit run — no auto-fallback, you compare both afterward and pick a winner by hand. Also writes to Drive.

In [ ]:
model_s = YOLO('yolo11s.pt')
results_s = model_s.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    project=str(DRIVE_RUNS_ROOT / 'yolo11s'),
    name='s_baseline',
    device=0,
)
s_best = results_s.save_dir / 'weights' / 'best.pt'
print('yolo11s best weights (on Drive):', s_best)

files.download(str(s_best))

## (Optional, later) Compare on the held-out test split
Mirrors `training/evaluate.py` (mAP50, mAP50-95, precision, recall, GPU ms/image here — CPU inference speed still needs to be checked separately on your own machine, since that's what the deployed service will actually use). Needs both `n_best` and `s_best` from the cells above.

In [ ]:
import time

def evaluate(weights_path, label):
    m = YOLO(str(weights_path))
    metrics = m.val(data=str(yaml_path), split='test')
    test_images = list((DATA_ROOT/'images/test').glob('*'))
    t0 = time.perf_counter()
    m.predict(source=str(DATA_ROOT/'images/test'), verbose=False)
    gpu_ms = (time.perf_counter() - t0) * 1000 / max(1, len(test_images))
    return {
        'model': label,
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'gpu_ms_per_image': gpu_ms,
    }

rows = [evaluate(n_best, 'yolo11n'), evaluate(s_best, 'yolo11s')]
headers = list(rows[0].keys())
widths = {h: max(len(h), *(len(f"{r[h]}") for r in rows)) for h in headers}
print(' | '.join(h.ljust(widths[h]) for h in headers))
print('-+-'.join('-'*widths[h] for h in headers))
for r in rows:
    print(' | '.join(f"{r[h]}".ljust(widths[h]) for h in headers))